In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pymc3 as pm
import math
from numpy import linalg
from datetime import datetime, timedelta
from scipy.interpolate import CubicSpline
from scipy.interpolate import pchip
import matplotlib.pyplot as plt

The version of PyMC you are using is very outdated.

Please upgrade to the latest version of PyMC https://www.pymc.io/projects/docs/en/stable/installation.html

Also notice that PyMC3 has been renamed to PyMC.


In [3]:
#Load temperature data
trap_test = 'sample'
trap_station = 'sample'
tempdata=pd.read_csv("data/temp_2016-2019.csv",low_memory=False)
tempdataframe=pd.DataFrame(tempdata)
tempdataframe.set_index('TRAPID')
templist1 = tempdataframe.loc[tempdataframe['TRAPID']==trap_station]
templist1 =np.array(templist1)
templist1 = np.delete(templist1,[0,1,2,3,4,5,6,7,8,9],None)
templist1 = templist1.astype(float)

tempdata2=pd.read_csv("data/temp_2020-2022.csv",low_memory=False)
tempdataframe2=pd.DataFrame(tempdata2)
tempdataframe2.set_index('TRAPID')
templist2=tempdataframe2.loc[tempdataframe['TRAPID']==trap_station]
templist2=np.array(templist2)
templist2=np.delete(templist2,[0,1,2,3,4,5,6,7,8,9],None)
templist2=templist2.astype(float)

templist=[]
for i in range(0,len(templist1)):
    templist.append(templist1[i])
for i in range(0,len(templist2)):
    templist.append(templist2[i])
    
for i in range(0,len(templist)):
    if math.isnan(templist[i])==True:
        templist[i]=templist[i-1]
        
#Take temperature average
templist_ave=[]
for i in range(1,int(len(templist)/8)):
    templist_ave.append(np.mean(templist[(i-1)*8:i*8]))
    
templist_lw=[]
for i in range(0,7):
    templist_lw.append(templist_ave[i])
for i in range(7,len(templist_ave)):
    templist_lw.append(np.mean(templist_ave[i-7:i]))
    
# Define the thermal dynamic functions for two stage model, the curves are directly taken from Mordecai 2017
def ovirate(T):
    c = 8.56*10**(-3)
    T0 = 14.58
    Tm = 34.61
    if T<T0:
        return 0.01
    elif T>Tm:
        return 0.01 
    else:
        return np.max([0.01,c*T*(T-T0)*(Tm-T)**0.5])


def pEA(T):
    c = -5.99*10**(-3)
    T0 = 13.56
    Tm = 38.29
    if T<T0:
        return 0.01
    elif T>Tm:
        return 0.01
    else:
        return np.max([0.01,c*(T-T0)*(T-Tm)])
    
    
def develop(T):
    c = 7.86*10**(-5)
    T0 = 11.36
    Tm = 39.17
    if T<T0:
        return 0.01
    elif T>Tm:
        return 0.01 
    else:
        return np.max([0.01,c*T*(T-T0)*(Tm-T)**0.5])

def adudeath(T):
    c = -1.48*10**(-1)
    T0 = 9.16
    Tm = 37.73
    if T<T0:
        return 1
    elif T>Tm:
        return 1
    else:
        return 1/(c*(T-T0)*(T-Tm))
    
def tempfunc(t):
    return (t-32)*5/9

#Define the time-dependent parameters in the two-dimensional ODE model
develop_list = []
ovi_list = []
adudeath_list = []
pEA_list = []
for i in range(0,len(templist_lw)):
    develop_list.append(develop(tempfunc(templist_lw[i])))
    pEA_list.append(pEA(tempfunc(templist_lw[i])))
    ovi_list.append(ovirate(tempfunc(templist_lw[i])))
    adudeath_list.append(adudeath(tempfunc(templist_lw[i])))
    
#Make the time-dependent parameters continuously differentiable
xs = np.linspace(0,len(templist_lw)-1,len(templist_lw))
cs_pos_develop = pchip(xs,develop_list)
cs_pos_ovi = pchip(xs,ovi_list)
cs_pos_adudeath = pchip(xs,adudeath_list)
cs_pos_pEA = pchip(xs,pEA_list)

#Read and prepare the mosquito trap data (aggregated data)
mosdata=pd.read_csv("data/2019_2023_mosq_agg_aedes.csv",low_memory=False)
mosdata=pd.DataFrame(mosdata)
datalist = mosdata['Ae_aegypti'][0:36]
timelist = []
for i in range(1,13):
    delta = datetime(2019,i,15)-datetime(2016,8,1)
    timelist.append(float(delta.days))
for i in range(1,13):
    delta = datetime(2020,i,15)-datetime(2016,8,1)
    timelist.append(float(delta.days))
for i in range(1,13):
    delta = datetime(2021,i,15)-datetime(2016,8,1)
    timelist.append(float(delta.days))
    
cs_pos = pchip(timelist,datalist)

#Use the Inverse Method to generate time-dependent carrying capacity
def JJsign(t):
    return cs_pos(t,1)/cs_pos_adudeath(t)+cs_pos(t)

xs_obtime = np.linspace(timelist[0],timelist[-1],int(timelist[-1]-timelist[0])+1)

find_list = []
for i in range(0,len(xs_obtime)):
    find_list.append(JJsign(xs_obtime[i]))
    
find_max = -min(find_list)

Acorr = max(0,2*np.ceil(find_max))

def JJcorr(t):
    return 2*(cs_pos(t,1)+cs_pos_adudeath(t)*(cs_pos(t)+Acorr))/cs_pos_develop(t)/cs_pos_pEA(t)
def Jprime(t):
    return 2*(cs_pos(t,2)+cs_pos_adudeath(t,1)*(cs_pos(t)+Acorr)+cs_pos_adudeath(t)*cs_pos(t,1))/cs_pos_develop(t)/cs_pos_pEA(t)-2*(cs_pos_develop(t,1)*cs_pos_pEA(t)+cs_pos_develop(t)*cs_pos_pEA(t,1))*(cs_pos(t,1)+cs_pos_adudeath(t)*(cs_pos(t)+Acorr))/cs_pos_develop(t)/cs_pos_develop(t)/cs_pos_pEA(t)/cs_pos_pEA(t)
def netgrowth(t):
    return Jprime(t)+cs_pos_develop(t)*JJcorr(t)
def directbirth(t):
    return cs_pos_ovi(t)*(cs_pos(t)+Acorr)

def capacityK_inv(t):
    return (cs_pos_ovi(t)*(cs_pos(t)+Acorr)-Jprime(t)-cs_pos_develop(t)*JJcorr(t))/(cs_pos_ovi(t)*JJcorr(t)*(cs_pos(t)+Acorr))

#Plot the estimated carrying capacity
series = pd.Series(capacityK_inv(xs_obtime))

rolling_avg = series.rolling(window=7).mean()

plt.figure(figsize=(6, 4))
ax = plt.gca()
plt.plot(xs_obtime[7:],1/rolling_avg[7:]/max(1/rolling_avg[7:]),color='#708090')
plt.axvspan(timelist[0], timelist[4], color='#F4A460', alpha=0.3, label='Dry Season')
plt.axvspan(timelist[4], timelist[9], color='#ADD8E6', alpha=0.3, label='Wet Season')
plt.axvspan(timelist[9], timelist[16], color='#F4A460', alpha=0.3)
plt.axvspan(timelist[16], timelist[21], color='#ADD8E6', alpha=0.3)
plt.axvspan(timelist[21], timelist[28], color='#F4A460', alpha=0.3)
plt.axvspan(timelist[28], timelist[33], color='#ADD8E6', alpha=0.3)
plt.legend(loc='upper center')
plt.title('Carrying Capacity for Aggregated Ae. aegypti Data')
ax.set_xticks([timelist[4], timelist[9],timelist[16],timelist[21],timelist[28],timelist[33]])
ax.set_xticklabels(['19May', '19Oct','20May', '20Oct', '21May', '21Oct'])
plt.tight_layout()